In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("../data/raw/spotify-tracks-dataset-detailed.csv")

audio_features = [
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence",
    "tempo",
    "loudness",
    "duration_ms",
    "key",
    "mode",
    "time_signature"
]

X = df[audio_features].copy()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=audio_features,
    index=df.index
)

X_scaled.head()

In [ ]:
pca = PCA()

X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(explained_variance) + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(0.90, linestyle="--", label="90% variance")
plt.axhline(0.95, linestyle="--", label="95% variance")

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=audio_features,
    columns=[
        f"PC{i+1}"
        for i in range(len(audio_features))
    ]
)

loadings

In [ ]:
loadings[["PC1", "PC2", "PC3"]].sort_values(
    "PC1",
    key=lambda x: x.abs(),
    ascending=False
)

In [ ]:
n_components = 8

pca_final = PCA(n_components=n_components)

X_pca = pca_final.fit_transform(X_scaled)

print(
    f"Explained variance with {n_components} components: "
    f"{pca_final.explained_variance_ratio_.sum():.2%}"
)

In [ ]:
pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(n_components)]
)

plt.figure(figsize=(10, 7))

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    alpha=0.15,
    s=10
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Songs in PCA Feature Space")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = pca_df.copy()
plot_df["genre"] = df["track_genre"].values

top_genres = df["track_genre"].value_counts().head(8).index

plot_df_filtered = plot_df[
    plot_df["genre"].isin(top_genres)
]

plt.figure(figsize=(11, 8))

sns.scatterplot(
    data=plot_df_filtered,
    x="PC1",
    y="PC2",
    hue="genre",
    alpha=0.5,
    s=25
)

plt.title("PCA Space Colored by Major Genres")
plt.tight_layout()
plt.show()